# Leakage-aware Evaluation of Infant Cry Classification — Colab runner

Runs the three core experiments of the paper (`multiclass`, `binary`, `leakage`) from Google Colab, with data and repo living on Google Drive.

**Use a GPU runtime**: `Runtime > Change runtime type > T4 GPU` (or better) before running anything below. All three scripts fall back to CPU/scikit-learn automatically if no GPU is available, but that's slow — see the compute-backend note at the bottom.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Get the repo

Two options — pick ONE. Default below is the real path for this project (repo already lives in Drive, inside `Mi unidad/Publicaciones/Chillanto CIARP/`). If you're reusing this notebook for a different Drive layout, adjust `REPO_DIR` or use Option B to clone fresh instead.

In [ ]:
# Option A (default): repo already present in Drive, at its real path for this project.
REPO_DIR = "/content/drive/MyDrive/Publicaciones/Chillanto CIARP/leakage-aware-infant-cry-classification"
%cd "$REPO_DIR"

# Option B: clone fresh into the Colab runtime instead (uncomment and set REPO_URL).
# REPO_URL = "https://github.com/pantrok/leakage-aware-infant-cry-classification.git"
# REPO_DIR = "/content/leakage-aware-infant-cry-classification"
# !git clone "$REPO_URL" "$REPO_DIR"
# %cd "$REPO_DIR"

## 3. Point to the dataset

See [`data/README.md`](../data/README.md) for the expected folder layout (`1s_asphyxia/`, `1s_deaf/`, `1s_hunger/`, `1s_normal/`, `1s_pain/`). The dataset itself is not distributed in this repo.

For this project the real data already lives in Drive, in a sibling folder to the repo (`Bebes Grid indiv - Optuna - 2/Data/`) rather than copied into `data/dataset/` — Option A below points straight at it so nothing has to be duplicated. If you'd rather keep a copy inside `data/dataset/` (e.g. for a different machine/Drive layout), use Option B instead.

In [ ]:
import os

# Option A (default): point directly at the real data folder already in Drive, outside the repo.
DATA_DIR = "/content/drive/MyDrive/Publicaciones/Chillanto CIARP/Bebes Grid indiv - Optuna - 2/Data"

# Option B: use a copy placed inside the repo instead (uncomment).
# DATA_DIR = os.path.join(REPO_DIR, "data", "dataset")

expected = ["1s_asphyxia", "1s_deaf", "1s_hunger", "1s_normal", "1s_pain"]
missing = [d for d in expected if not os.path.isdir(os.path.join(DATA_DIR, d))]
assert not missing, (
    f"DATA_DIR '{DATA_DIR}' is missing expected subfolders: {missing}. "
    "Place the Baby Chillanto 1s_* folders there before continuing "
    "(see data/README.md)."
)
print("Dataset layout OK:", DATA_DIR)

## 4. Install dependencies

`requirements.txt` covers everything needed for the CPU/scikit-learn path (which is also what reproduces the paper's exact numbers). `cuml-cu12` is the optional GPU accelerator for SVM/kNN/RandomForest — install it too if you want the speed-up on Colab's GPU runtime; PyTorch (used for the GPU MLP) already ships with Colab, nothing extra needed for that part.

In [ ]:
%pip install -r requirements.txt

In [ ]:
# Optional but recommended given the GPU runtime: cuML (RAPIDS) for SVM/kNN/RandomForest.
# Skip this cell to stay on scikit-learn/CPU for those classifiers (still reproduces the
# paper's numbers exactly; cuML is not binarily identical to sklearn -- see
# src/gpu_classifiers.py's docstring). Takes a few minutes to install.
%pip install --extra-index-url=https://pypi.nvidia.com cuml-cu12

## 5. (Recommended) Sanity check: recording-ID grouping

Confirms the file-name based grouping used by the `leakage` mode's `G-1s` condition is working as expected before committing to a long run. Should print `M (grabaciones distintas, total) = 73`.

In [ ]:
!python scripts/count_recordings.py --data_dir "$DATA_DIR"

## 6. Run the three experiments

Flags shown are the scripts' own defaults (100 Optuna trials, 5 seeds, 5-fold CV, `k_best_d=60`, `fv_n_components=16`) — the same ones used to produce `results/*.csv` in this repo. Lower `--n_trials`/`--n_seeds` for a quick smoke test before committing to a full run.

Each script prints a `[INFO] Backend de cómputo: ...` line at startup (via `report_backend()`) confirming whether it's actually using cuML/PyTorch-GPU or scikit-learn/CPU for that run — check it before assuming the GPU is being used.

In [ ]:
# Multiclass (Tables 2-3): feature-set x classifier Optuna search
!python main.py multiclass --data_dir "$DATA_DIR" --n_trials 100 --n_seeds 5 --n_splits 5 --balanced \
    --csv_summary results/optuna_summary_multiclass.csv --csv_history results/optuna_history_multiclass.csv

In [ ]:
# Binary (Table 4): healthy vs pathology
!python main.py binary --data_dir "$DATA_DIR" --n_trials 100 --n_seeds 5 --n_splits 5 --balanced \
    --csv_summary results/binary_summary.csv --csv_history results/binary_history.csv

In [ ]:
# Leakage comparison (Section 4.3, Tables 5-6, Figures 6-8): S-1s vs G-1s.
# This is the expensive one even with GPU classifiers, since feature extraction
# (audio load + FOSP/EEFGabor/Fisher Vector) runs once per feature set on CPU
# regardless. Run in a cell by itself so Colab doesn't disconnect it as "idle"
# while it's still computing (interact with the tab occasionally, or use a
# Colab Pro background execution session for very long runs).
!python main.py leakage --data_dir "$DATA_DIR" --n_seeds 5 --n_splits 5 --balanced \
    --csv_results results/leakage_results.csv --csv_stats results/leakage_stats.csv --plot_dir results/leakage_plots

## Notes on compute backend and expected runtime

All three scripts (`optuna_search.py`, `optuna_binary.py`, `leakage_comparison.py`) now build their classifiers through `src/gpu_classifiers.py`: cuML (RAPIDS) for SVM/kNN/RandomForest and PyTorch+CUDA for the MLP when available, with an automatic fallback to scikit-learn/CPU otherwise — that CPU/scikit-learn fallback is what reproduces the exact figures already in `results/`. `cuML` is not binarily identical to scikit-learn (different internal solvers/initializations, documented in `src/gpu_classifiers.py`'s own docstring), so numbers from a GPU run may differ slightly from the CPU baseline already in this repo; that's expected, not a bug, and is exactly the kind of thing to flag back rather than silently reconcile.

`cuML` has no native Windows build (Linux/WSL2 or Colab only) — this is precisely why this rerun needs to happen in Colab rather than on a Windows desktop.

**Time budget**: a small-scale CPU-only smoke test of `leakage_comparison.py` (`--feature_sets B --classifiers MLP --n_splits 2 --n_seeds 1`, a small fraction of a full run) took about 65 minutes. Classifier fitting is only part of that cost — feature extraction (audio load, band-pass filtering, FOSP/EEFGabor/Fisher Vector) happens once per feature set and is CPU-bound regardless of GPU classifiers, so GPU acceleration helps but does not eliminate the time cost of a full run. Budget accordingly, and consider splitting the full grid across multiple sessions with `--feature_sets`/`--classifiers`/`--modes` if a free Colab session (idle-disconnects after ~90 min, hard cap around 12h) isn't enough.